In [127]:
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
    RunnableSequence,
)
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from youtube_transcript_api import TranscriptsDisabled, YouTubeTranscriptApi

load_dotenv()

True

In [128]:
vid_id= 'fsLh-NYhOoU'
api= YouTubeTranscriptApi()
try:
    transcript_list= api.fetch(vid_id)
    
    transcript= ' '.join(chunk.text for chunk in transcript_list)
    print(transcript)
    
except TranscriptsDisabled:
    print("No captions available in this video")

[Submit subtitle corrections at criblate.com] Thank you very much. It is good to be here. I don't know if you people realize what a beautiful campus you have and how you basically just study in heaven. Today, I want to talk with you about what I think is one of the most underappreciated formulas, not because those who know it don't appreciate it, but because not enough people know about this. And more importantly, not enough people understand where it comes from. And this should be one of the gems. It's this should be the e to the pi i of the mathematical community. But I really want you to come away knowing not just what it is, but why it's true and what it represents. Now, before I dive straight into it, I think the scene is best set if we start with two different puzzles. The first puzzle is just going to give us a sense of the meaning of what we're about to do because it runs the risk of feeling pretty abstract. And then the second puzzle is going to be a kind of forewarning, not t

# Step-by-Step

## Splitting transcripts

In [129]:
splitter= RecursiveCharacterTextSplitter(
    chunk_size= 1000,
    chunk_overlap= 100
)

chunks= splitter.create_documents([transcript])

In [130]:
len(chunks)

78

In [131]:
chunks[0]

Document(metadata={}, page_content="[Submit subtitle corrections at criblate.com] Thank you very much. It is good to be here. I don't know if you people realize what a beautiful campus you have and how you basically just study in heaven. Today, I want to talk with you about what I think is one of the most underappreciated formulas, not because those who know it don't appreciate it, but because not enough people know about this. And more importantly, not enough people understand where it comes from. And this should be one of the gems. It's this should be the e to the pi i of the mathematical community. But I really want you to come away knowing not just what it is, but why it's true and what it represents. Now, before I dive straight into it, I think the scene is best set if we start with two different puzzles. The first puzzle is just going to give us a sense of the meaning of what we're about to do because it runs the risk of feeling pretty abstract. And then the second puzzle is goin

## Embedding and string transcripts in vector store

In [132]:
embedding_model= MistralAIEmbeddings()
vector_store= FAISS.from_documents(chunks, embedding_model)

In [133]:
vector_store.index_to_docstore_id

{0: 'e3019ae5-c0fc-4855-8504-77aea7952e01',
 1: '9e7e17f3-384b-456a-bb06-1ead0bfe877b',
 2: '95a0b1ef-a316-4b79-a4bc-6ed86ea2ab8d',
 3: '03e84d22-ea90-490f-aab6-59b1db0f8abc',
 4: '0d0819d8-f422-48f2-b202-c265b3d13ad6',
 5: 'ced92886-4d8e-475e-9fe2-270ee147a820',
 6: 'cedafbc6-3515-4837-b0eb-1a19e42ccddd',
 7: 'd3dca79a-df63-41f9-9c74-ab17d84b0c42',
 8: '1967eea7-e59e-43d7-bf5b-5607a07c6e59',
 9: 'bcd1d705-bf58-4aaa-9099-92e922f0a4aa',
 10: '673f7846-b749-4491-9b5a-30db62902b53',
 11: 'bbe5cb0a-e7ff-480e-a6a9-88ff8cc0c7c9',
 12: 'd6633599-2225-48d2-bf8d-86aa20205278',
 13: '0a8bbd1a-806b-4c39-b6e7-da1d2446b48a',
 14: 'a807b406-60d7-4dc4-a790-163443536ea1',
 15: 'b4e0ef1a-2d7c-4354-8546-d528ccce10f5',
 16: '8c3b3aad-cef7-4c9a-b35a-e6632aa40f71',
 17: '2c1b5b0c-8943-40cd-9391-3b1955d6cf46',
 18: '622366fa-cb8c-4e9c-9548-7c0e28fce85b',
 19: 'aa04f02d-7b96-4199-bf3b-fb3a1f0f006e',
 20: '885e8ae3-8023-4bae-b03d-ed11a7ef54fe',
 21: '5b0a3c7e-df48-4b6a-a541-a1ca44fc7beb',
 22: '0bdd1ee3-cc41-

In [134]:
vector_store.get_by_ids(['75d0bdac-5f00-410e-acc6-221a5570e7b8'])

[]

## Retriever

In [135]:
retriever= vector_store.as_retriever(search_type= 'mmr', search_kwargs= {'k': 10, 'lambda_mult': 0.5})

In [136]:
retriever

VectorStoreRetriever(tags=['FAISS', 'MistralAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E4AE15CC50>, search_type='mmr', search_kwargs={'k': 10, 'lambda_mult': 0.5})

In [137]:
retriever.invoke('What is the general formula?')

[Document(id='e3c62df7-f352-4969-b0e4-5acf2e33f0e5', metadata={}, page_content="this two pi jump up into the right and then an integration factor to get down. And we just fill in the whole diagram and in some sense, we're done. In some sense, this gives you the rule that you could use to decide on the volume of any dimensional sphere that you want. But of course, we want a nice formula for this. We want to come away with formula. We want to analyze the formula. We want to think about what it means. So let's take a moment to see if we can write down what that should be. So the way I'm going to do this, each one of these volumes in higher dimensions looks like some constant times r to that number of dimensions. So for example, in one dimension, that constant is two. In two dimensions, that constant is pi. In three dimensions, four thirds pi, on and on and on. What we really want to know is what is this constant. And the key rule that we have is a recursive one. It's saying if you want to

## Augmentation (Query + retrieved documents + A prompt)

In [138]:
llm = ChatMistralAI(model='mistral-medium-latest', temperature=0.3)

In [139]:
prompt= PromptTemplate(
    template= """ 
    You are a helpful assistant. You provide answers without any font formatting. 
    Answer the questions from the provided transcript context only. 
    Reply 'insufficient knowledge or data not available in the video' if you don't know the answer.
    
    Context: {context}
    Question: {question}
    """,
    input_variables=['context', 'question']
)

In [140]:
question= 'Is the topic of aliens discussed in this video? If yes, then what was discussed?'
retrieved_docs= retriever.invoke(question)
context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)



In [141]:
final_prompt= prompt.invoke({'context': context_text, 'question': question})

## Final Answer

In [142]:
answer= llm.invoke(final_prompt)
print(answer.content)

insufficient knowledge or data not available in the video


# Complete Chain

In [143]:
parser= StrOutputParser()

def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

In [144]:
parallel_chain= RunnableParallel({
    'context': RunnableSequence(retriever, RunnableLambda(format_docs)),
    'question': RunnablePassthrough()
})

In [145]:
main_chain= RunnableSequence(parallel_chain, prompt, llm, parser)

In [146]:
result= main_chain.invoke('What is the name of the speaker?')
print(result)

insufficient knowledge or data not available in the video
